# Method 3 — Unsupervised Fellegi–Sunter Mixture Estimate

Estimates precision, recall, and per-criterion discriminative power of the M0 linkage rule using only the *distribution* of comparison vectors over all candidate pairs $\Omega$ (linked and unlinked) — no manual labels.

**Population.** $\Omega$ = all 6,137 candidate pairs (not just `candidate_rank == 1`). Rank-1 pairs are each source's *best* candidate by construction, so restricting to them would sample "winners" rather than "compared pairs" and would distort the apparent $U$-class (non-match) distribution. The full candidate table is the realized output of blocking (same `buyer_key`, temporal window around `estimated_end_date`) and is the correct Fellegi-Sunter comparison space, exactly analogous to classical record linkage where $\Omega$ is all blocked pairs, not just the arg-max per block.

**Model.** Two latent classes $M$ (match/renewal) and $U$ (non-match), four comparison dimensions $\gamma = (s_{text}, s_{cpv}, s_{time}, s_{buyer})$, conditionally independent given class. Each dimension is Beta-distributed per class ($s_{text}$/$s_{time}$ are continuous in $[0,1]$; $s_{cpv}$/$s_{buyer}$ are ordinal/multi-valued rather than binary — modeling all four as Beta avoids an arbitrary binarization cutoff and keeps a single unified model):

$$P(\gamma \mid M) = \prod_d \mathrm{Beta}(\gamma_d; \alpha_{M,d}, \beta_{M,d}), \qquad P(\gamma \mid U) = \prod_d \mathrm{Beta}(\gamma_d; \alpha_{U,d}, \beta_{U,d}), \qquad P(\gamma) = p\,P(\gamma\mid M) + (1-p)\,P(\gamma\mid U)$$

Because $\gamma$ contains exact 0s and 1s (Beta density is undefined at the boundary), each dimension is transformed with the Smithson–Verkuilen squeeze $\gamma_d' = \gamma_d(1-2\epsilon) + \epsilon$, $\epsilon = 1/(2N)$, before fitting.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist
from scipy.special import logsumexp
from scipy.stats import pearsonr, spearmanr

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLES_DIR = PROJECT_ROOT / "reports" / "tables"
TABLES_DIR.mkdir(parents=True, exist_ok=True)

DIMS = ["s_text", "s_cpv", "s_time", "s_buyer"]
MAX_ITER = 200
TOL = 1e-6
N_RANDOM_RESTARTS = 5
RANDOM_SEED_BASE = 0

## EM algorithm

**E-step** (log-space, for numerical stability):
$$\log\text{num}_{M,j} = \log p + \sum_d \log\mathrm{Beta.pdf}(\gamma'_{j,d};\alpha_{M,d},\beta_{M,d}), \qquad r_j = P(M\mid\gamma_j) = \exp\big(\log\text{num}_{M,j} - \mathrm{logsumexp}(\log\text{num}_{M,j},\log\text{num}_{U,j})\big)$$

**M-step** (weighted method-of-moments, per class $c$, per dimension $d$):
$$m_{c,d} = \frac{\sum_j w_j\gamma'_{j,d}}{\sum_j w_j}, \qquad v_{c,d} = \frac{\sum_j w_j(\gamma'_{j,d}-m_{c,d})^2}{\sum_j w_j}, \qquad \alpha_{c,d} = m_{c,d}\Big(\tfrac{m_{c,d}(1-m_{c,d})}{v_{c,d}}-1\Big), \quad \beta_{c,d} = (1-m_{c,d})\Big(\tfrac{m_{c,d}(1-m_{c,d})}{v_{c,d}}-1\Big)$$

**Why moment-matching instead of `scipy.stats.beta.fit` (weighted MLE):** (a) scipy's Beta MLE fitter has no native support for per-observation responsibility weights — using it inside EM would require resampling by weight (adds Monte Carlo noise every iteration) or a hand-rolled weighted Newton/digamma fixed-point solve (Minka's method), substantially more code and a second, less stable source of non-convergence; (b) MLE Beta fitting is numerically fragile exactly where this data is worst-behaved: $s_{cpv}$ and $s_{buyer}$ are near-degenerate (few distinct values — $s_{buyer}$ is empirically only 2-valued in this corpus, $\{0.6, 1.0\}$), and Newton-Raphson on the digamma equations can diverge or walk to a boundary for spiky/discrete-like data; (c) weighted moment-matching is closed-form, always well-defined given the variance floor below, and is standard practice for Beta-mixture EM on bounded scores (e.g. methylation / ChIP-seq beta-mixture models).

**A genuine degeneracy, found and fixed.** An unregularized first run of this EM converged to a solution with log-likelihood $71{,}478.2$ in which one component isolated exactly the pairs with $s_{buyer}=1.0$ (`RAW_SIRET`-keyed sources) and drove its $s_{buyer}$ variance toward zero — the classical unbounded-likelihood degenerate-component failure of finite mixture EM, triggered here because $s_{buyer}$ is only 2-valued. Flooring every class-conditional variance at 5% of $\Omega$'s unconditional per-dimension variance (and clipping the mean away from $\{0,1\}$ so the floor is always satisfiable) removes this spurious optimum — implemented as `var_floor` / `m_min` below.

In [ ]:
def squeeze(x: np.ndarray, eps: float) -> np.ndarray:
    """Smithson-Verkuilen squeeze into the open interval (0, 1)."""
    return x * (1 - 2 * eps) + eps


def logpdf_beta_matrix(X: np.ndarray, alphas: np.ndarray, betas: np.ndarray) -> np.ndarray:
    """Sum of per-dimension Beta log-densities for one class (conditional independence)."""
    out = np.zeros(X.shape[0])
    for d in range(X.shape[1]):
        out += beta_dist.logpdf(X[:, d], alphas[d], betas[d])
    return out


def m_step_moment_match(X, w, var_floor, m_min):
    """Weighted method-of-moments M-step (see markdown above for the derivation
    and the degeneracy fix this variance floor / mean clip exists to prevent)."""
    wsum = w.sum()
    m = (w[:, None] * X).sum(axis=0) / wsum
    m = np.clip(m, m_min, 1 - m_min)
    v = (w[:, None] * (X - m[None, :]) ** 2).sum(axis=0) / wsum
    v = np.clip(v, var_floor, 0.99 * m * (1 - m))
    common = m * (1 - m) / v - 1
    common = np.maximum(common, 1e-6)
    alpha = m * common
    beta_ = (1 - m) * common
    return alpha, beta_, m, v


def run_em(X, r0, label, var_floor, m_min):
    """Runs EM to convergence from initial responsibilities r0 = P(M | gamma)."""
    r = r0.copy()
    prev_ll = -np.inf

    for it in range(MAX_ITER):
        p = r.mean()
        p = np.clip(p, 1e-6, 1 - 1e-6)
        aM, bM, mM, vM = m_step_moment_match(X, r, var_floor, m_min)
        aU, bU, mU, vU = m_step_moment_match(X, 1 - r, var_floor, m_min)

        log_num_M = np.log(p) + logpdf_beta_matrix(X, aM, bM)
        log_num_U = np.log(1 - p) + logpdf_beta_matrix(X, aU, bU)
        log_denom = logsumexp(np.vstack([log_num_M, log_num_U]), axis=0)
        r = np.exp(log_num_M - log_denom)
        r = np.clip(r, 1e-12, 1 - 1e-12)

        ll = log_denom.sum()
        if it > 0 and abs(ll - prev_ll) < TOL:
            break
        prev_ll = ll

    print(f"  [{label}] converged after {it + 1} iterations, final logL = {ll:.4f}")
    return {
        "label": label, "p": p,
        "alpha_M": aM, "beta_M": bM, "mean_M": mM, "var_M": vM,
        "alpha_U": aU, "beta_U": bU, "mean_U": mU, "var_U": vU,
        "r": r, "loglik": ll, "n_iter": it + 1,
    }

## Load $\Omega$, squeeze, and fit the regularization floor

In [ ]:
print("Loading candidate pairs (Omega)...")
pairs = pd.read_csv(PROCESSED_DIR / "boamp_m0_candidate_pairs.csv")
N = len(pairs)
print(f"Omega size (all candidate pairs): {N}")

balanced = pd.read_csv(PROCESSED_DIR / "boamp_m0_links_balanced.csv")
balanced_keys = set(zip(balanced["source_notice_id"], balanced["candidate_notice_id"]))
pairs["in_balanced_linked_set"] = [
    (s, c) in balanced_keys
    for s, c in zip(pairs["source_notice_id"], pairs["candidate_notice_id"])
]
n_balanced = pairs["in_balanced_linked_set"].sum()
print(f"Balanced-linked pairs found within Omega: {n_balanced} (expect {len(balanced)})")
assert n_balanced == len(balanced), "balanced links must all be present in the candidate table"

eps = 1.0 / (2 * N)
X = np.column_stack([squeeze(pairs[d].to_numpy(dtype=float), eps) for d in DIMS])

# Regularization floor: no component may claim less than 5% of Omega's
# unconditional per-dimension spread; mean is clipped away from {0,1}
# accordingly. This is what rules out the degenerate single-point-mass
# solution described above (s_buyer is only 2-valued in this corpus).
REG_FRAC = 0.05
global_var = X.var(axis=0)
var_floor = REG_FRAC * global_var
m_min = 0.5 - np.sqrt(np.maximum(0.25 - var_floor, 0.0))
print(f"Global (unconditional) variance per dimension: {dict(zip(DIMS, np.round(global_var, 5)))}")
print(f"Variance floor (5% of global var): {dict(zip(DIMS, np.round(var_floor, 6)))}")

## Fit: a balanced-linkage warm start plus 5 random restarts, keep the best log-likelihood

In [ ]:
runs = []

# Warm start: responsibilities seeded from balanced-linked membership
# (0.9/0.1, not hard 0/1, so the first M-step is not degenerate).
r0_warm = np.where(pairs["in_balanced_linked_set"].to_numpy(), 0.9, 0.1)
runs.append(run_em(X, r0_warm, "warm_start", var_floor, m_min))

rng_master = np.random.RandomState(RANDOM_SEED_BASE)
for k in range(N_RANDOM_RESTARTS):
    r0_rand = rng_master.uniform(0.05, 0.95, size=N)
    runs.append(run_em(X, r0_rand, f"random_restart_{k}", var_floor, m_min))

max_ab = max(
    max(run["alpha_M"].max(), run["beta_M"].max(), run["alpha_U"].max(), run["beta_U"].max())
    for run in runs
)
if max_ab > 500:
    print(f"WARNING: at least one run has an extreme alpha/beta (max={max_ab:.1f}) "
          f"even after regularization - inspect the params table before trusting it.")

In [ ]:
best = max(runs, key=lambda d: d["loglik"])
warm = runs[0]
print(f"\nBest run: {best['label']} (logL={best['loglik']:.4f})")

# Convergence-robustness note: compare warm-start vs. best-overall.
warm_hard = warm["r"] > 0.5
best_hard = best["r"] > 0.5
inter = np.logical_and(warm_hard, best_hard).sum()
union = np.logical_or(warm_hard, best_hard).sum()
jaccard_warm_vs_best = inter / union if union else float("nan")
ll_delta_warm_vs_best = best["loglik"] - warm["loglik"]
print(f"Warm-start vs best-restart: loglik delta = {ll_delta_warm_vs_best:.4f}, "
      f"Jaccard(r>0.5) = {jaccard_warm_vs_best:.4f}")

# Post-hoc label-switching fix: "M" = component with the larger weighted
# mean of s_text + s_cpv (the two content-based signals).
idx_text, idx_cpv = DIMS.index("s_text"), DIMS.index("s_cpv")
comp0_content = best["mean_M"][idx_text] + best["mean_M"][idx_cpv]
comp1_content = best["mean_U"][idx_text] + best["mean_U"][idx_cpv]
if comp1_content > comp0_content:
    print("Swapping M/U labels post-hoc (component 1 has more content-signal mass).")
    best["p"] = 1 - best["p"]
    (best["alpha_M"], best["alpha_U"]) = (best["alpha_U"], best["alpha_M"])
    (best["beta_M"], best["beta_U"]) = (best["beta_U"], best["beta_M"])
    (best["mean_M"], best["mean_U"]) = (best["mean_U"], best["mean_M"])
    (best["var_M"], best["var_U"]) = (best["var_U"], best["var_M"])
    best["r"] = 1 - best["r"]

p_hat = best["p"]
r = best["r"]
gaps = best["mean_M"] - best["mean_U"]
gap_rank = np.argsort(-np.abs(gaps))

print(f"\np_hat = {p_hat:.4f}")
for d in gap_rank:
    print(f"  {DIMS[d]:8s}: mean_M={best['mean_M'][d]:.4f}  mean_U={best['mean_U'][d]:.4f}"
          f"  gap={gaps[d]:+.4f}")

## Write outputs: fitted params, per-pair posteriors, precision/recall, conditional-independence check

In [ ]:
param_rows = [{
    "class": "mixing_weight", "dimension": "p_hat", "alpha": np.nan, "beta": np.nan,
    "weighted_mean": p_hat, "weighted_var": np.nan, "gap_m_minus_u": np.nan,
    "gap_rank": np.nan,
}]
for rank_pos, d in enumerate(gap_rank):
    for cls, a_arr, b_arr, m_arr, v_arr in [
        ("M", best["alpha_M"], best["beta_M"], best["mean_M"], best["var_M"]),
        ("U", best["alpha_U"], best["beta_U"], best["mean_U"], best["var_U"]),
    ]:
        param_rows.append({
            "class": cls, "dimension": DIMS[d],
            "alpha": a_arr[d], "beta": b_arr[d],
            "weighted_mean": m_arr[d], "weighted_var": v_arr[d],
            "gap_m_minus_u": gaps[d], "gap_rank": rank_pos + 1,
        })
params_df = pd.DataFrame(param_rows)
params_df.to_csv(TABLES_DIR / "m3_beta_mixture_params.csv", index=False)
print(f"Wrote {TABLES_DIR / 'm3_beta_mixture_params.csv'}")

conv_row = pd.DataFrame([{
    "best_run_label": best["label"], "best_loglik": best["loglik"],
    "warm_start_loglik": warm["loglik"],
    "loglik_delta_warm_vs_best": ll_delta_warm_vs_best,
    "jaccard_warm_vs_best_r_gt_half": jaccard_warm_vs_best,
    "n_restarts_total": len(runs),
}])
conv_row.to_csv(TABLES_DIR / "m3_em_convergence_check.csv", index=False)

post_df = pairs[["source_notice_id", "candidate_notice_id", "candidate_rank",
                  "m0_composite_score", "in_balanced_linked_set"]].copy()
post_df["posterior_M"] = r
post_df.to_csv(TABLES_DIR / "m3_posteriors.csv", index=False)
print(f"Wrote {TABLES_DIR / 'm3_posteriors.csv'}")

params_df

In [ ]:
in_link = pairs["in_balanced_linked_set"].to_numpy()
TP_hat = r[in_link].sum()
FN_hat = r[~in_link].sum()
n_linked = int(in_link.sum())
precision_hat = TP_hat / n_linked
recall_hat = TP_hat / (TP_hat + FN_hat)
pr_df = pd.DataFrame([{
    "TP_hat": TP_hat, "FN_hat": FN_hat, "n_linked_balanced": n_linked,
    "n_omega": N, "precision_hat": precision_hat, "recall_hat": recall_hat,
    "scope_caveat": (
        "Recall_hat is defined relative to Omega (all 6,137 candidate pairs, "
        "i.e. pairs surviving buyer_key + temporal-window blocking) only. It "
        "structurally excludes the 1,923 of 3,159 eligible sources that had zero "
        "candidates in blocking, so it cannot bound recall against all possible "
        "true renewals - only against renewals that could in principle have been "
        "found given the current blocking rule."
    ),
}])
pr_df.to_csv(TABLES_DIR / "m3_precision_recall.csv", index=False)
print(f"Precision_hat = {precision_hat:.4f}  Recall_hat = {recall_hat:.4f}")
print(f"Wrote {TABLES_DIR / 'm3_precision_recall.csv'}")
pr_df

In [ ]:
linked_pairs = pairs[pairs["in_balanced_linked_set"]]
pear_r, pear_p = pearsonr(linked_pairs["s_cpv"], linked_pairs["s_text"])
spear_r, spear_p = spearmanr(linked_pairs["s_cpv"], linked_pairs["s_text"])
ci_df = pd.DataFrame([{
    "n_linked_pairs": len(linked_pairs),
    "pearson_r_cpv_text": pear_r, "pearson_p_cpv_text": pear_p,
    "spearman_r_cpv_text": spear_r, "spearman_p_cpv_text": spear_p,
    "flag_high_correlation_gt_0_3": bool(abs(pear_r) > 0.3 or abs(spear_r) > 0.3),
}])
ci_df.to_csv(TABLES_DIR / "m3_conditional_independence_check.csv", index=False)
print(f"Conditional-independence check (s_cpv vs s_text among balanced links): "
      f"pearson r={pear_r:.4f}, spearman r={spear_r:.4f}")
if abs(pear_r) > 0.3 or abs(spear_r) > 0.3:
    print("WARNING: conditional-independence assumption looks violated (|r| > 0.3) "
          "- a known bias source for Fellegi-Sunter mixture estimates; not "
          "corrected here, only flagged per the method spec.")
ci_df

## Result (from the last full run of this notebook)

$\hat p = 0.1809$. Most discriminative criterion by class gap: $s_{buyer}$ ($+0.3202$, a mechanical artifact since $s_{buyer}$ is constant per source, not a genuine per-pair signal), then $s_{cpv}$ ($+0.1385$) and $s_{text}$ ($+0.0979$) — the two genuinely pairwise, content-based signals — then $s_{time}$ ($-0.0109$, no separation). $\widehat{\text{Precision}} = 0.4552$, $\widehat{\text{Recall}} = 0.2534$ (relative to $\Omega$ only). Conditional-independence check between $s_{cpv}$ and $s_{text}$ among the 618 balanced-linked pairs: Pearson $r=0.0465$, Spearman $\rho=-0.1056$, both below the $0.3$ flag threshold. See `reports/linkage_quality_evaluation.tex` §2 for the full write-up.